# Homework №7 |  Long Video Generation with Causal Transformers and KV Caching

In this homework, you are asked to implement long video generation using a pretrained few-step autoregressive video diffusion model, [Self-Forcing](https://arxiv.org/pdf/2506.08009) (based on [Wan2.1-T2V-1.3B](https://github.com/Wan-Video/Wan2.1)).

**The model was trained to:** 
* Generate a chunk of 3 latent frames in parallel using few-step diffusion sampling
* Autoregressively generate videos with the context of 21 latent frames (5s videos)

---

### Task outline (3 pt)

1) Parallel few-step diffusion sampling **(0.25 pt)** (largely adopted from HWs 4 and 5)
2) Chunk-by-chunk autoregressive video generation with KV caching **(2 pt)**
3) Rolling KV-caching for longer video generation **(0.75 pt)**


In [ ]:
!pip install -q ftfy

## Preparing the model and prompt embeddings

We precalculate the prompt embeddings for you since the text encoder (`T5-XXL`) takes too much memory.

**Precalculated prompt list**

**0:** `A stylish woman strolls down a bustling Tokyo street, the warm glow of neon lights and animated city signs casting vibrant reflections...`

**1:** `A stunning mid-afternoon landscape photograph with a low camera angle, showcasing several giant wooly mammoths treading through a snowy meadow...`

**2:** `A movie trailer in a classic cinematic style, featuring the adventurous journey of a 30-year-old space man wearing a vibrant red wool knitted motorcycle helmet...`

**3:** `A drone view of waves crashing against the rugged cliffs along Big Sur’s Garay Point beach...`

**4:** `A close-up 3D animated scene of a short, fluffy monster kneeling beside a melting red candle...`


In [ ]:
import os
from huggingface_hub import snapshot_download

# Download base model architecture + weights
snapshot_download(
    "Wan-AI/Wan2.1-T2V-1.3B",
    allow_patterns=["config.json", "diffusion_pytorch_model.safetensors"],
    local_dir="wan_models/Wan2.1-T2V-1.3B/",
)
print("Base model downloaded.")

# Download VAE weights
os.makedirs("checkpoints", exist_ok=True)
if not os.path.exists("checkpoints/taew2_1.pth"):
    !wget -q -O checkpoints/taew2_1.pth https://github.com/madebyollin/taehv/raw/main/taew2_1.pth
print("TAEHV downloaded.")

# Download Self-Forcing distilled checkpoint
snapshot_download(
    "gdhe17/Self-Forcing",
    allow_patterns=["checkpoints/self_forcing_dmd.pt"],
    local_dir=".",
)

# Download prompt_embed
if not os.path.exists("prompt_embeds.pt"):
    !wget -q https://storage.yandexcloud.net/yandex-research/visual-genai/prompt_embeds.pt
print("prompt_embeds downloaded.")

In [ ]:
import torch
import gc
from einops import rearrange
from diffusers.utils import export_to_video

from utils.wan_wrapper import WanDiffusionWrapper
from taehv import TAEHV

torch.set_grad_enabled(False)
device = torch.device("cuda")
dtype = torch.float16

In [ ]:
# Load the causal transformer model
print("Loading transformer model...")
generator = WanDiffusionWrapper(
    timestep_shift=5.0,
    is_causal=True,
    local_attn_size=-1,
).to(device=device, dtype=dtype)

state_dict = torch.load("checkpoints/self_forcing_dmd.pt", map_location="cpu")
generator.load_state_dict(state_dict['generator_ema'])
del state_dict
gc.collect()

generator = generator.eval()
generator.model.num_frame_per_chunk = 3

print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# Load TAEHV VAE decoder
class TAEHVDecoder(torch.nn.Module):
    def __init__(self, checkpoint_path):
        super().__init__()
        self.taehv = TAEHV(checkpoint_path).to(dtype)

    def decode(self, latents):
        pixels = self.taehv.decode_video(latents, parallel=False)
        return pixels.mul_(2).sub_(1)

vae = TAEHVDecoder("checkpoints/taew2_1.pth").to(device)
print(f"VAE loaded. GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# Load prompts text for display
with open("prompts.txt") as f:
    PROMPTS = [line.strip() for line in f]

# Load precomputed prompt embeddings
prompt_embeds_all = torch.load("prompt_embeds.pt", map_location="cpu")
print(f"Loaded {prompt_embeds_all.shape[0]} prompt embeddings, shape: {prompt_embeds_all.shape}")

PROMPT_INDEX = 4
prompt_embeds = prompt_embeds_all[PROMPT_INDEX:PROMPT_INDEX+1].to(device=device, dtype=dtype)
conditional_dict = {"prompt_embeds": prompt_embeds}
print(f"\nPrompt [{PROMPT_INDEX}]: {PROMPTS[PROMPT_INDEX]}")

In [ ]:
# ==================== Constants ====================
NUM_FRAMES = 21
LATENT_C = 16
LATENT_H = 60
LATENT_W = 104
FRAME_SEQ_LEN = 1560     # tokens per frame: (60/2) * (104/2)
NUM_TRANSFORMER_BLOCKS = 30
NUM_HEADS = 12
HEAD_DIM = 128
BATCH_SIZE = 1

# ==================== Scheduler ====================
scheduler = generator.get_scheduler()
raw_steps = torch.tensor([1000, 750, 500, 250], dtype=torch.long)
timesteps_table = torch.cat((scheduler.timesteps.cpu(), torch.tensor([0.0])))
denoising_step_list = timesteps_table[1000 - raw_steps]
print(f"Denoising timesteps: {denoising_step_list.tolist()}")

# ==================== Helpers ====================
def make_noise(num_frames, seed=0):
    rng = torch.Generator(device=device).manual_seed(seed)
    return torch.randn(
        [BATCH_SIZE, num_frames, LATENT_C, LATENT_H, LATENT_W],
        device=device, dtype=dtype, generator=rng
    )

def decode_and_save(latents, filename="output.mp4", fps=16):
    from diffusers.video_processor import VideoProcessor
    with torch.no_grad():
        pixels = vae.decode(latents)
    pixels = (pixels * 0.5 + 0.5).clamp(0, 1)
    video = rearrange(pixels, 'b t c h w -> b c t h w').cpu()
    processor = VideoProcessor()
    video_frames = processor.postprocess_video(video=video * 2 - 1, output_type="pil")
    export_to_video(video_frames[0], filename, fps=fps)
    print(f"Saved {filename} ({latents.shape[1]} latent frames, {pixels.shape[3]}x{pixels.shape[4]} px)")
    return pixels

def display_video(source):
    from IPython.display import HTML, display
    from base64 import b64encode

    if isinstance(source, str) and source.startswith(("http://", "https://")):
        html = f'<video controls autoplay loop style="max-width:640px"><source src="{source}" type="video/mp4"></video>'
    else:
        with open(source, "rb") as f:
            data = b64encode(f.read()).decode()
        html = f'<video controls autoplay loop style="max-width:640px"><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'
    display(HTML(html))

---
## Parallel Few-step Sampling (0.25 pt)

Generate a chunk of 3 latent frames in parallel using 4-step **stochastic multistep sampling** (from HWs 4 and 5), where the sampling steps are specified by `denoising_step_list`.

In [ ]:
def multistep_sampling(noisy_input, generator, conditional_dict, scheduler, denoising_step_list,
                       kv_cache=None, crossattn_cache=None, current_start=0, rng=None, seed=0):
    """
    Run stochastic multistep sampling (aka consistency sampling) on a chunk of frames.

    Args:
        noisy_input: [B, T, C, H, W] noisy latent frames
        generator: WanDiffusionWrapper model
        conditional_dict: {"prompt_embeds": [B, 512, 4096]}
        scheduler: FlowMatchScheduler
        denoising_step_list: tensor of timestep values defined above
        kv_cache: optional list of cache dicts for self-attention
        crossattn_cache: optional list of cache dicts for cross-attention
        current_start: token offset for positional encoding (blockwise only)
        rng: optional torch.Generator for re-noising (passed by generate_blockwise)
        seed: random seed to create rng if rng is None

    Returns:
        denoised: [B, T, C, H, W] denoised latent frames
    """
    if rng is None:
        rng = torch.Generator(device=device).manual_seed(seed)

    batch_size, num_frames = noisy_input.shape[0], noisy_input.shape[1]

    # ===== YOUR CODE HERE =====
    # Implement the multi-step denoising loop. 
    # 1) timestep shape should be [batch_size, num_frames]
    # 2) when noising the latents, use torch.randn instead of randn_like, 
    #    since you cannot pass torch.Generator (rng) to it.
    # 3) For re-noising (q_sample), use "scheduler.add_noise()""

    raise NotImplementedError("TODO: Implement the multistep sampling")

    # ===== END YOUR CODE =====

    return denoised_pred

In [ ]:
noise = make_noise(num_frames=3, seed=5)

torch.cuda.reset_peak_memory_stats()
output_parallel = multistep_sampling(noise, generator, conditional_dict, scheduler, denoising_step_list, seed=5)
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

In [ ]:
decode_and_save(output_parallel, "parallel_3_latent_frames.mp4")
display_video("parallel_3_latent_frames.mp4")

### Reference video (3 latent frames in parallel)

In [ ]:
display_video("https://storage.yandexcloud.net/yandex-research/visual-genai/task1_parallel.mp4")

In [ ]:
del output_parallel, noise
gc.collect()
torch.cuda.empty_cache()

---
## Chunk-by-Chunk Autoregressive Generation with KV Cache (2 pt)

The model can generate videos of 3 latent frames in parallel but we want longer videos. Let's generate 5-second videos autoregressively.

### The challenge

When generating chunk-by-chunk, the current chunk needs to attend to previously generated frames. But those frames were processed in earlier forward passes - their attention keys and values are gone. We need a **Key-Value (KV) cache** to save and reuse them.

**KV-cache** allows saving compute by storing previously calculated attention keys and values, which remain unchanged over the autoregressive sampling of future chunks. 

### Cross-attention cache (0.5 pt)

The text prompt is the same for every chunk, so the cross-attention keys and values can be **precomputed once** and reused. You will implement `compute_crossattn_cache()` by iterating over the model's transformer blocks and extracting each block's cross-attention K and V projections.

Each block has a `cross_attn` sub-layer with:
- `cross_attn.k` - key projection (nn.Linear)
- `cross_attn.v` - value projection (nn.Linear)
- `cross_attn.norm_k` - key normalization (RMSNorm)
- `cross_attn.num_heads`, `cross_attn.head_dim` - for reshaping

### Self-attention KV cache (hook-based) (0.5 pt)

We've added a hook mechanism to the model's self-attention layers.

The handler receives:
- `roped_key`: current block's keys with RoPE positional encoding applied, shape `[B, L_block, num_heads, head_dim]`
- `v`: current block's values (no positional encoding on values), shape `[B, L_block, num_heads, head_dim]`

The handler must return `(full_k, full_v)` for attention - which should include cached entries from previous blocks concatenated with the current block's entries.

#### Two modes

Your self-attention cache needs two modes:
- **Read mode** (during denoising steps): attend to cached context from previous blocks, but do NOT save to cache (because the current input is noisy/intermediate)
- **Write mode** (during context caching step): save the clean denoised features to the cache

After denoising each chunk, we re-run the model with **timestep=0** on the clean output in write mode.


In [ ]:
@torch.no_grad()
def compute_crossattn_cache(model, prompt_embeds):
    """
    Precompute cross-attention keys and values for all transformer blocks.

    Args:
        model: the CausalWanModel (generator.model)
        prompt_embeds: [B, 512, 4096] text embeddings

    Returns:
        list of dicts [{"k": ..., "v": ...}, ...], one per transformer block
    """
    prompt_embeds = model.text_embedding(prompt_embeds)

    cache = []
    # ===== YOUR CODE HERE =====
    # For each model block, compute *normalized* keys and values
    # and reshape them to [B, -1, num_heads, head_dim]

    raise NotImplementedError("TODO: Implement cross-attention cache computation")

    # ===== END YOUR CODE =====

    return cache

In [ ]:
class KVCache:
    """
    Hook-based KV cache for chunk-by-chunk generation.

    Attaches handler functions to the model's attention layers via the
    kv_cache["_handler"] mechanism.

    Modes:
        saving=False (read mode): prepend cached K,V to current K,V for attention
        saving=True  (write mode): save current K,V to cache, then return full cache

    Args:
        num_layers: number of transformer blocks (30)
        max_tokens: if set, remove oldest entries when cache exceeds this size
    """

    def __init__(self, num_layers, max_tokens=None):
        self.num_layers = num_layers
        self.max_tokens = max_tokens
        self.cache_k = [None] * num_layers
        self.cache_v = [None] * num_layers
        self.saving = False

    def get_handler(self, layer_idx):
        """Return a handler callable for one transformer layer.

        The handler receives:
            roped_k: [B, L_block, num_heads, head_dim] - current keys (with RoPE)
            v:       [B, L_block, num_heads, head_dim] - current values

        Must return:
            (full_k, full_v) - keys and values for attention, including cached entries
        """
        def handler(roped_k, v):
            cached_k = self.cache_k[layer_idx]
            cached_v = self.cache_v[layer_idx]

            if self.saving:
                # ===== YOUR CODE HERE =====
                # In WRITE mode, append roped_k to cached_k and v to cached_v,
                # after that return the full cache as (full_k, full_v)
                #
                # Remember that cache can be empty (None)

                raise NotImplementedError("TODO: Implement KV cache handler (write mode)")
                
                if self.max_tokens is not None:
                    # ===== YOUR CODE HERE =====
                    # Delete the context tail if the context exceedes max_tokens: 
                    # after writing, keep only the last max_tokens tokens.
                   
                    raise NotImplementedError("TODO: Implement KV cache when max_tokens is given")

                    # ===== END YOUR CODE =====
            else:
                # ===== YOUR CODE HERE =====
                # in READ mode, concatenate cached and current and return (do NOT modify the cache)
                raise NotImplementedError("TODO: Implement KV cache handler (read mode)")

        return handler

    def make_kv_cache_list(self):
        """Create the kv_cache list-of-dicts that the model expects."""
        return [{"_handler": self.get_handler(i)} for i in range(self.num_layers)]

    def reset(self):
        """Clear all cached entries."""
        self.cache_k = [None] * self.num_layers
        self.cache_v = [None] * self.num_layers


### Implement chunkwise AR diffusion sampling (1 pt)

**Schematically:** Loop over chunks, where each chunk is generated with the few-step diffusion in parallel. At the same time, maintain the `cache` to provide the context about previously generated chunks. 

We describe the sampling algorithm in more details in the `generate_chunkwise` function.

In [ ]:
def generate_chunkwise(noise, generator, conditional_dict, scheduler,
                       denoising_step_list, num_frame_per_chunk=3,
                       max_cache_tokens=None, context_noise=0, seed=0):
    """
    Generate video chunk-by-chunk using KV cache.

    Args:
        noise: [B, T, C, H, W]
        generator: WanDiffusionWrapper
        conditional_dict: {"prompt_embeds": [B, 512, 4096]}
        scheduler: FlowMatchScheduler
        denoising_step_list: warped timestep values
        num_frame_per_chunk: frames per chunk (default 3)
        max_cache_tokens: if set, remove oldest tokens when cache exceeds this
        context_noise: timestep for context caching (default 0)
        seed: random seed for re-noising in denoise steps

    Returns:
        output: [B, T, C, H, W] denoised latent frames
    """
    rng = torch.Generator(device=device).manual_seed(seed)

    batch_size, num_frames = noise.shape[0], noise.shape[1]
    assert num_frames % num_frame_per_chunk == 0
    num_blocks = num_frames // num_frame_per_chunk

    output = torch.zeros_like(noise)

    # Create self-attention KV cache and cross-attention cache
    cache = KVCache(NUM_TRANSFORMER_BLOCKS, max_tokens=max_cache_tokens)
    kv_cache = cache.make_kv_cache_list()
    crossattn_cache = compute_crossattn_cache(generator.model, conditional_dict["prompt_embeds"])

    # ===== YOUR CODE HERE =====
    # Implement the chunk-by-chunk AR generation loop:
    #
    # current_start_frame = 0
    #
    # For each chunk:
    #   1. KV cache READ mode
    #   2. Generate a current chunk using multistep_sampling(). Remember to pass the caches,
    #      current_start=current_start_frame * FRAME_SEQ_LEN,
    #      and rng=rng (pass the shared rng to reproduce our reference videos)
    #   3. KV cache WRITE mode
    #   4. Cache clean KV by re-running the generator with the current clean chunk and t=0 
    #   5. Update current_start_frame

    raise NotImplementedError("TODO: Implement chunk-by-chunk AR generation with KV cache")

    # ===== END YOUR CODE =====

    del cache, kv_cache, crossattn_cache
    return output

In [ ]:
noise = make_noise(NUM_FRAMES, seed=5)

torch.cuda.reset_peak_memory_stats()
output_chunkwise = generate_chunkwise(
    noise, generator, conditional_dict, scheduler, denoising_step_list,
    num_frame_per_chunk=3, seed=5
)
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

In [ ]:
decode_and_save(output_chunkwise, "chunkwise_ar_21_latent_frames.mp4")
display_video("chunkwise_ar_21_latent_frames.mp4")

### Reference video (5-second video)

In [ ]:
display_video("https://storage.yandexcloud.net/yandex-research/visual-genai/task2_blockwise.mp4")

In [ ]:
del output_chunkwise, noise
gc.collect()
torch.cuda.empty_cache()

---
## Rolling KV Cache for Longer Video Generation (0.75 pt)

### Motivation: why can't we just generate more frames? (0.25 pt)

Previously, we generated 21 latent frames autoregressively chunk-by-chunk (5s video). What about generating 42 latent frames (10s video)? 

1) The KV cache stores keys and values for **all previously generated chunks at each attention layer** $\to$ **increased memory cost**

2) The model was trained with the maximum context length of 21 latent frames $\to$ **training-inference mismatch**:  generating with the longer context would result in artifacts since the model gets out-of-distribution inputs.

First, let's estimate the memory cost:


In [ ]:
# How much memory are we using right now?
print(f"Current GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU total:          {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

# ===== YOUR CODE HERE =====
# Estimate the KV cache size for 42 frames.

kv_bytes = ...  # TODO

# ===== END YOUR CODE =====

print(f"\nKV cache for 42 frames: {kv_bytes / 1024**3:.2f} GB")
print(f"Model + KV cache:       {torch.cuda.memory_allocated()/1024**3 + kv_bytes/1024**3:.2f} GB")
print(f"Remaining for activations: {torch.cuda.get_device_properties(0).total_memory/1024**3 - torch.cuda.memory_allocated()/1024**3 - kv_bytes/1024**3:.2f} GB")

In [ ]:
# (If GPU memory allows) Try generating 42 frames WITHOUT rolling cache.
# On a T4 (15 GB) this will likely OOM. In this case, please see the reference video. 
noise_42 = make_noise(42, seed=5)
torch.cuda.reset_peak_memory_stats()
try:
    output_42 = generate_chunkwise(
        noise_42, generator, conditional_dict, scheduler, denoising_step_list,
        num_frame_per_chunk=3, seed=5, # no max_cache_tokens!
    )
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
    pixels_t2 = decode_and_save(output_42, "task3_no_rolling.mp4")
    display_video("task3_no_rolling.mp4")
    del output_42
except torch.cuda.OutOfMemoryError:
    print("Out of memory! The unbounded KV cache is too large.")
gc.collect(); torch.cuda.empty_cache()

### Reference video (No rolling)

Please pay attention to the last generated seconds

In [ ]:
display_video("https://storage.yandexcloud.net/yandex-research/visual-genai/task3_no_rolling.mp4")

### Solution: rolling KV cache (0.5 pt)

A **rolling cache** removes the oldest entries when it exceeds a maximum size, bounding memory at a fixed level regardless of the video length.

Go back to the `KVCache` class and add the deletion logic in `get_handler()`. After writing new K,V to the cache, if `self.max_tokens` is set and the cache exceeds it, trim to the most recent `max_tokens` tokens.

Since `max_tokens` defaults to `None`, your previous code is unaffected -- trimming is activated when you explicitly pass `max_cache_tokens` to `generate_blockwise`.

```
After block 6:  cache = [fr 0..20]    (21 frames = 32760 tokens -- FULL)
After block 7:  cache = [fr 3..23]    (removed oldest 3 frames, added 3 new)
After block 8:  cache = [fr 6..26]    (removed oldest 3, added 3 new)
  ...
```


In [ ]:
NUM_EXTENDED_FRAMES = 42

noise = make_noise(NUM_EXTENDED_FRAMES, seed=5)

torch.cuda.reset_peak_memory_stats()
output_rolling = generate_chunkwise(
    noise, generator, conditional_dict, scheduler, denoising_step_list,
    num_frame_per_chunk=3, max_cache_tokens=21 * FRAME_SEQ_LEN, context_noise=0, seed=5
)
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

In [ ]:
decode_and_save(output_rolling, "rolling_kv_cache_10s.mp4")
display_video("rolling_kv_cache_10s.mp4")

### Reference video (Rolling)

In [ ]:
display_video("https://storage.yandexcloud.net/yandex-research/visual-genai/task3_extended.mp4")

In [ ]:
del output_rolling, noise
gc.collect()
torch.cuda.empty_cache()